# KoBERT 기반 2차 모델 학습 (순수 텍스트 기반)
## 맥락 기반 보이스피싱 탐지 모델 - 화자 정보 제거

## 1. 환경 설정 및 라이브러리 import

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. 데이터 로딩 및 전처리

In [2]:
# 훈련 데이터 로딩 (merged_labeled_data.csv)
train_df = pd.read_csv("../../dataset/master_dataset_final.csv")
print(f"훈련 데이터 수: {len(train_df)}")
print(f"피싱: {train_df['is_phishing'].sum()}, 일반: {len(train_df) - train_df['is_phishing'].sum()}")
print(f"고유 file_id 수: {train_df['file_id'].nunique()}")

# 테스트 데이터 로딩 (1차모델_테스트데이터셋.csv)
test_df = pd.read_csv("../../dataset/1차모델_테스트데이터셋.csv")
print(f"\n테스트 데이터 수: {len(test_df)}")
print(f"피싱: {test_df['is_phishing'].sum()}, 일반: {len(test_df) - test_df['is_phishing'].sum()}")

# 데이터 미리보기
print("\n=== 훈련 데이터 미리보기 ===")
print(train_df.head())
print("\n=== 테스트 데이터 미리보기 ===")
print(test_df.head())

훈련 데이터 수: 12000
피싱: 6000, 일반: 6000
고유 file_id 수: 12000

테스트 데이터 수: 1000
피싱: 500, 일반: 500

=== 훈련 데이터 미리보기 ===
         file_id                                               text  \
0    normal_2408  아 그럼 알겠습니다. 그러면 지금 바로 환불 접수 도와드릴까요? 에 좀 해주시겠어요...   
1    normal_0867  네 네 감사합니다. 예. o/입금일은 이번 주 금요일 오후 (6시)/(여섯 시) 이...   
2  phishing_2722  안녕하세요, OOO님. 국민연금관리공단의 정입니다. 네, 어떤 일이세요? '납부 내...   
3  phishing_1239  역과 국민계좌가 시중에서 새로 개술이 됐습니다. 네. 뭐 아시는 부분이라도 전혀 없...   
4    normal_0475  o/ 네. 장바구니 들어가시면은요n/. 거기 오른쪽 하단에 재 결제라는 버튼 보이시...   

  phishing_type  is_phishing  
0           NaN            0  
1           NaN            0  
2         기관사칭형            1  
3         기관사칭형            1  
4           NaN            0  

=== 테스트 데이터 미리보기 ===
  file_name                                               text  is_phishing
0         0  예 고객님 담당자 김성도 대리입니다.예지금 법무사님이 두분 배정되셨어요.네 네네 네...            1
1         2  6시 되가지고 전화 해봤습니다.예 예 그 앞전에 이면주 법무사님 영수증 확인되셨는데...            1
2         3  네 네 네 여보세요네 네어

In [3]:
# 개선된 대화 시퀀스 생성 (증강 제거 - 이미 균형잡힌 데이터)
import re

def clean_text(text):
    """텍스트 전처리 함수"""
    # 불필요한 공백 제거
    text = re.sub(r'\s+', ' ', text)
    # 특수문자 정리 (필요한 문장부호는 유지)
    text = re.sub(r'[^\w\s가-힣.,!?()]', '', text)
    return text.strip()

def create_dialogue_sequences_from_merged(df):
    dialogues = []
    
    for file_id in df['file_id'].unique():
        file_data = df[df['file_id'] == file_id]
        full_text = file_data['text'].iloc[0]
        
        # 개선된 문장 분할
        sentences = []
        for sent in full_text.split('.'):
            sent = clean_text(sent)
            if sent and len(sent.split()) >= 2:  # 최소 2단어 이상인 문장만
                sentences.append(sent)
        
        label = file_data['is_phishing'].iloc[0]
        
        if len(sentences) > 1:
            dialogues.append({
                'file_id': file_id,
                'texts': sentences,
                'label': label
            })
    
    return dialogues

def create_single_text_data(df):
    test_data = []
    
    for idx, row in df.iterrows():
        full_text = row['text']
        sentences = []
        
        for sent in full_text.split('.'):
            sent = clean_text(sent)
            if sent and len(sent.split()) >= 2:
                sentences.append(sent)
        
        if len(sentences) == 0:
            sentences = [clean_text(full_text)]
        
        test_data.append({
            'file_id': f"test_{idx}",
            'texts': sentences,
            'label': row['is_phishing']
        })
    
    return test_data

# 데이터 증강 없이 기본 대화 시퀀스만 생성
train_dialogues = create_dialogue_sequences_from_merged(train_df)
test_dialogues = create_single_text_data(test_df)

print(f"훈련 대화 시퀀스 수: {len(train_dialogues)}")
print(f"평균 문장 수: {np.mean([len(d['texts']) for d in train_dialogues]):.2f}")
print(f"생성된 테스트 데이터 수: {len(test_dialogues)}")

# 클래스 분포 확인
train_labels = [d['label'] for d in train_dialogues]
print(f"클래스 분포 - 일반: {train_labels.count(0)}, 피싱: {train_labels.count(1)}")
print(f"클래스 비율 - 일반:{train_labels.count(0)/len(train_labels):.1%}, 피싱:{train_labels.count(1)/len(train_labels):.1%}")

훈련 대화 시퀀스 수: 11683
평균 문장 수: 25.04
생성된 테스트 데이터 수: 1000
클래스 분포 - 일반: 5792, 피싱: 5891
클래스 비율 - 일반:49.6%, 피싱:50.4%


## 3. KoBERT 및 데이터셋 클래스 정의

In [4]:
MODEL_NAME = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
kobert_model = AutoModel.from_pretrained(MODEL_NAME)

class ImprovedDialogueDataset(Dataset):
    def __init__(self, dialogues, tokenizer, max_length=128, max_turns=50):
        self.dialogues = dialogues
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_turns = max_turns
    
    def __len__(self):
        return len(self.dialogues)
    
    def __getitem__(self, idx):
        dialogue = self.dialogues[idx]
        texts = dialogue['texts'][:self.max_turns]
        label = dialogue['label']
        
        input_ids_list = []
        attention_mask_list = []
        
        for text in texts:
            text = str(text).strip()
            if len(text) == 0:
                text = "[EMPTY]"
            
            # 특수 토큰 추가로 문맥 구분
            text = f"[TURN] {text}"
            
            encoded = self.tokenizer(
                text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))
        
        return {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'label': torch.tensor(label, dtype=torch.long),
            'num_turns': len(texts)
        }

def improved_collate_fn(batch):
    max_turns = max([item['num_turns'] for item in batch])
    
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []
    batch_lengths = []
    
    for item in batch:
        num_turns = item['num_turns']
        
        if num_turns < max_turns:
            pad_size = max_turns - num_turns
            pad_input_ids = torch.zeros(pad_size, item['input_ids'].size(1), dtype=torch.long)
            pad_attention_mask = torch.zeros(pad_size, item['attention_mask'].size(1), dtype=torch.long)
            
            input_ids = torch.cat([item['input_ids'], pad_input_ids], dim=0)
            attention_mask = torch.cat([item['attention_mask'], pad_attention_mask], dim=0)
        else:
            input_ids = item['input_ids']
            attention_mask = item['attention_mask']
        
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(item['label'])
        batch_lengths.append(num_turns)
    
    return {
        'input_ids': torch.stack(batch_input_ids),
        'attention_mask': torch.stack(batch_attention_mask),
        'labels': torch.stack(batch_labels),
        'lengths': torch.tensor(batch_lengths, dtype=torch.long)
    }

print("개선된 데이터셋 클래스 정의 완료")

개선된 데이터셋 클래스 정의 완료


## 4. 텍스트 전용 모델 정의

In [5]:
class EnhancedPhishingDetector(nn.Module):
    def __init__(self, kobert_model, hidden_size=512, num_classes=2, dropout=0.3, num_heads=8):
        super(EnhancedPhishingDetector, self).__init__()
        
        self.kobert = kobert_model
        self.kobert_hidden_size = kobert_model.config.hidden_size
        
        # 개선된 문장 임베딩 레이어
        self.sentence_projection = nn.Sequential(
            nn.Linear(self.kobert_hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # 양방향 LSTM with 더 많은 레이어
        self.dialogue_lstm = nn.LSTM(
            hidden_size, 
            hidden_size // 2, 
            num_layers=2,  # 레이어 수 증가
            batch_first=True, 
            bidirectional=True,
            dropout=dropout
        )
        
        # Multi-head Attention with residual connection
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        
        self.attention_norm = nn.LayerNorm(hidden_size)
        
        # 위치 인코딩 추가
        self.positional_encoding = nn.Parameter(
            torch.randn(100, hidden_size) * 0.02  # 최대 100턴
        )
        
        # 개선된 분류기
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout // 2),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.LayerNorm(hidden_size // 4),
            nn.ReLU(),
            nn.Dropout(dropout // 2),
            nn.Linear(hidden_size // 4, num_classes)
        )
        
        # Focal Loss를 위한 파라미터
        self.alpha = nn.Parameter(torch.tensor(0.25))
        self.gamma = nn.Parameter(torch.tensor(2.0))
        
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
    
    def forward(self, input_ids, attention_mask, lengths):
        batch_size, max_turns, seq_len = input_ids.size()
        
        # BERT 임베딩 (gradient 허용)
        input_ids_flat = input_ids.view(-1, seq_len)
        attention_mask_flat = attention_mask.view(-1, seq_len)
        
        kobert_outputs = self.kobert(
            input_ids=input_ids_flat,
            attention_mask=attention_mask_flat
        )
        
        sentence_embeddings = kobert_outputs.last_hidden_state[:, 0, :]
        sentence_embeddings = sentence_embeddings.view(batch_size, max_turns, -1)
        
        # 문장 특성 추출
        sentence_features = self.sentence_projection(sentence_embeddings)
        
        # 위치 인코딩 추가
        pos_encoding = self.positional_encoding[:max_turns].unsqueeze(0).expand(batch_size, -1, -1)
        sentence_features = sentence_features + pos_encoding
        
        # LSTM 처리
        lstm_out, _ = self.dialogue_lstm(sentence_features)
        
        # Attention with residual connection
        max_len = max_turns
        padding_mask = torch.arange(max_len, device=lengths.device).expand(
            batch_size, max_len
        ) >= lengths.unsqueeze(1)
        
        attended_out, attention_weights = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=padding_mask
        )
        
        # Residual connection + Layer Norm
        attended_out = self.attention_norm(lstm_out + attended_out)
        
        # 마스킹 및 풀링
        mask = ~padding_mask.unsqueeze(-1)
        masked_attended = attended_out * mask
        dialogue_repr = masked_attended.sum(dim=1) / lengths.unsqueeze(-1).float()
        
        # 분류
        logits = self.classifier(dialogue_repr)
        
        return {
            'logits': logits,
            'attention_weights': attention_weights,
            'dialogue_repr': dialogue_repr
        }

# 향상된 모델 초기화
model = EnhancedPhishingDetector(kobert_model, hidden_size=512, dropout=0.2)
model.to(device)

print(f"향상된 모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print("향상된 아키텍처로 모델 초기화 완료")

향상된 모델 파라미터 수: 97,003,652
향상된 아키텍처로 모델 초기화 완료


## 5. K-Fold 설정 및 훈련 함수

In [6]:
file_ids = [d['file_id'] for d in train_dialogues]
labels = [d['label'] for d in train_dialogues]

test_dataset = ImprovedDialogueDataset(test_dialogues, tokenizer)

# 향상된 하이퍼파라미터
K_FOLDS = 5
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
BATCH_SIZE = 16  # 배치 크기 증가
LEARNING_RATE = 1e-5  # 더 작은 학습률
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

print(f"훈련 대화 수: {len(train_dialogues)}")
print(f"테스트 데이터: {len(test_dialogues)}")
print(f"K-Fold 수: {K_FOLDS}")
print(f"배치 크기: {BATCH_SIZE}, 학습률: {LEARNING_RATE}")

훈련 대화 수: 11683
테스트 데이터: 1000
K-Fold 수: 5
배치 크기: 16, 학습률: 1e-05


In [7]:
# Focal Loss 구현
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Early Stopping 구현
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model)
        else:
            self.counter += 1
            
        if self.counter >= self.patience:
            if self.restore_best_weights:
                model.load_state_dict(self.best_weights)
            return True
        return False
    
    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

def enhanced_train_epoch(model, train_loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['lengths'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, lengths)
        logits = outputs['logits']
        
        loss = criterion(logits, labels)
        loss.backward()
        
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        if scheduler:
            scheduler.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), 100. * correct / total

def enhanced_validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            lengths = batch['lengths'].to(device)
            
            outputs = model(input_ids, attention_mask, lengths)
            logits = outputs['logits']
            
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            probs = torch.softmax(logits, dim=1)
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return total_loss / len(val_loader), 100. * correct / total, all_predictions, all_labels, all_probs

print("향상된 훈련 및 검증 함수 정의 완료")

향상된 훈련 및 검증 함수 정의 완료


In [ ]:
# 클래스 가중치 계산 함수 (개선됨)
def calculate_enhanced_class_weights(labels):
    """
    데이터 불균형을 해결하기 위한 향상된 클래스 가중치 계산
    """
    from sklearn.utils.class_weight import compute_class_weight
    import numpy as np
    
    unique_classes = np.unique(labels)
    class_weights = compute_class_weight(
        'balanced', 
        classes=unique_classes, 
        y=labels
    )
    
    class_weight_dict = dict(zip(unique_classes, class_weights))
    print(f"클래스 분포: {np.bincount(labels)}")
    print(f"클래스 가중치: {class_weight_dict}")
    
    # PyTorch tensor로 변환
    weight_tensor = torch.FloatTensor([class_weights[0], class_weights[1]])
    return weight_tensor

# 앙상블을 위한 모델 클래스
class EnsemblePhishingDetector:
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights if weights else [1.0 / len(models)] * len(models)
    
    def predict(self, input_ids, attention_mask, lengths):
        all_probs = []
        
        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(input_ids, attention_mask, lengths)
                probs = torch.softmax(outputs['logits'], dim=1)
                all_probs.append(probs)
        
        # 가중 평균
        ensemble_probs = sum(w * p for w, p in zip(self.weights, all_probs))
        return ensemble_probs

# 전체 훈련 데이터의 클래스 가중치 계산
all_labels = np.array(labels)
class_weights = calculate_enhanced_class_weights(all_labels)

print(f"적용될 클래스 가중치: Normal={class_weights[0]:.3f}, Phishing={class_weights[1]:.3f}")

print("향상된 K-Fold 교차검증 시작...")

fold_results = []
all_val_accs = []
fold_models = []  # 앙상블을 위한 모델 저장

for fold, (train_indices, val_indices) in enumerate(skf.split(file_ids, labels)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{K_FOLDS} 시작")
    print(f"{'='*50}")
    
    fold_train_dialogues = [train_dialogues[i] for i in train_indices]
    fold_val_dialogues = [train_dialogues[i] for i in val_indices]
    
    print(f"Fold {fold+1} - 훈련: {len(fold_train_dialogues)}, 검증: {len(fold_val_dialogues)}")
    
    # 현재 fold의 클래스 분포 확인 및 가중치 계산
    fold_train_labels = [fold_train_dialogues[i]['label'] for i in range(len(fold_train_dialogues))]
    fold_class_weights = calculate_enhanced_class_weights(np.array(fold_train_labels))
    
    fold_train_dataset = ImprovedDialogueDataset(fold_train_dialogues, tokenizer)
    fold_val_dataset = ImprovedDialogueDataset(fold_val_dialogues, tokenizer)
    
    fold_train_loader = DataLoader(
        fold_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=improved_collate_fn
    )
    fold_val_loader = DataLoader(
        fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=improved_collate_fn
    )
    
    fold_model = EnhancedPhishingDetector(kobert_model, hidden_size=512, dropout=0.2).to(device)
    
    # Focal Loss 적용
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    
    # AdamW 옵티마이저 with 향상된 파라미터
    optimizer = optim.AdamW(
        fold_model.parameters(), 
        lr=LEARNING_RATE, 
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )
    
    # 학습률 스케줄러 (Cosine Annealing with Warm Restart)
    total_steps = len(fold_train_loader) * 10  # 10 에포크
    warmup_steps = int(total_steps * WARMUP_RATIO)
    
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=total_steps//3, eta_min=1e-7
    )
    
    # Early Stopping
    early_stopping = EarlyStopping(patience=3, min_delta=0.001)
    
    fold_train_accs = []
    fold_val_accs = []
    best_val_acc = 0
    
    NUM_EPOCHS = 10  # 에포크 수 증가
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = enhanced_train_epoch(
            fold_model, fold_train_loader, criterion, optimizer, scheduler, device
        )
        val_loss, val_acc, val_preds, val_labels, val_probs = enhanced_validate_epoch(
            fold_model, fold_val_loader, criterion, device
        )
        
        fold_train_accs.append(train_acc)
        fold_val_accs.append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        # Early stopping 체크
        if early_stopping(val_loss, fold_model):
            print(f"Early stopping at epoch {epoch+1}")
            break
        
        if epoch % 2 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%, LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    print(f"Fold {fold+1} 최고 검증 정확도: {best_val_acc:.2f}%")
    
    # 모델 저장 (앙상블용)
    fold_models.append(fold_model.cpu())
    fold_model.to(device)  # 다시 GPU로
    
    fold_results.append({
        'fold': fold+1,
        'best_val_acc': best_val_acc,
        'final_val_acc': fold_val_accs[-1],
        'train_accs': fold_train_accs,
        'val_accs': fold_val_accs,
        'val_predictions': val_preds,
        'val_labels': val_labels,
        'val_probs': val_probs
    })
    all_val_accs.append(best_val_acc)

# K-Fold 결과 요약
print(f"\n{'='*60}")
print("향상된 K-Fold 교차검증 결과 요약")
print(f"{'='*60}")

for i, result in enumerate(fold_results):
    print(f"Fold {i+1}: {result['best_val_acc']:.2f}%")

mean_acc = np.mean(all_val_accs)
std_acc = np.std(all_val_accs)

print(f"\n평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"최고 검증 정확도: {max(all_val_accs):.2f}%")
print(f"최저 검증 정확도: {min(all_val_accs):.2f}%")

# 앙상블 모델 생성
ensemble_model = EnsemblePhishingDetector(fold_models)
print(f"\n{len(fold_models)}개 모델로 앙상블 생성 완료")

클래스 분포: [5792 5891]
클래스 가중치: {np.int64(0): np.float64(1.008546270718232), np.int64(1): np.float64(0.9915973518927177)}
적용될 클래스 가중치: Normal=1.009, Phishing=0.992
향상된 K-Fold 교차검증 시작...

Fold 1/5 시작
Fold 1 - 훈련: 9346, 검증: 2337
클래스 분포: [4633 4713]
클래스 가중치: {np.int64(0): np.float64(1.0086337146557307), np.int64(1): np.float64(0.9915128368342881)}


In [ ]:
print(f"\n{'='*50}")
print("전체 데이터로 최종 모델 훈련 및 앙상블 테스트")
print(f"{'='*50}")

# 최종 모델 훈련 (단일 모델)
final_train_dataset = ImprovedDialogueDataset(train_dialogues, tokenizer)
final_train_loader = DataLoader(
    final_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=improved_collate_fn
)

final_model = EnhancedPhishingDetector(kobert_model, hidden_size=512, dropout=0.2).to(device)

# Focal Loss 적용
criterion = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = optim.AdamW(
    final_model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)

# 학습률 스케줄러
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=len(final_train_loader)*3, eta_min=1e-7
)

NUM_FINAL_EPOCHS = 8
for epoch in range(NUM_FINAL_EPOCHS):
    train_loss, train_acc = enhanced_train_epoch(
        final_model, final_train_loader, criterion, optimizer, scheduler, device
    )
    if epoch % 2 == 0:
        print(f"Final Epoch {epoch+1}/{NUM_FINAL_EPOCHS} - Train Acc: {train_acc:.2f}%, LR: {optimizer.param_groups[0]['lr']:.2e}")

# 테스트 데이터 평가 (단일 모델)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=improved_collate_fn
)

print(f"\n단일 모델 테스트 평가 시작... (테스트 샘플 수: {len(test_dialogues)})")

final_model.eval()
test_correct = 0
test_total = 0
test_predictions = []
test_labels = []
test_probabilities = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing Single Model'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['lengths'].to(device)
        
        outputs = final_model(input_ids, attention_mask, lengths)
        logits = outputs['logits']
        
        probs = torch.softmax(logits, dim=1)
        
        _, predicted = torch.max(logits.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        test_predictions.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probabilities.extend(probs.cpu().numpy())

single_model_accuracy = 100. * test_correct / test_total

# 앙상블 모델 테스트
print(f"\n앙상블 모델 테스트 평가 시작...")

ensemble_predictions = []
ensemble_probabilities = []

# 모든 fold 모델을 GPU로 이동
for model in fold_models:
    model.to(device)

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing Ensemble'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['lengths'].to(device)
        
        # 앙상블 예측
        ensemble_probs = ensemble_model.predict(input_ids, attention_mask, lengths)
        
        _, predicted = torch.max(ensemble_probs.data, 1)
        ensemble_predictions.extend(predicted.cpu().numpy())
        ensemble_probabilities.extend(ensemble_probs.cpu().numpy())

ensemble_accuracy = 100. * sum(p == l for p, l in zip(ensemble_predictions, test_labels)) / len(test_labels)

print(f"\n=== 최종 테스트 결과 비교 ===")
print(f"K-Fold 평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"단일 모델 테스트 정확도: {single_model_accuracy:.2f}%")
print(f"앙상블 모델 테스트 정확도: {ensemble_accuracy:.2f}%")
print(f"앙상블 개선 효과: {ensemble_accuracy - single_model_accuracy:+.2f}%")

# 단일 모델 성능 분석
print(f"\n=== 단일 모델 상세 성능 ===")
print(classification_report(test_labels, test_predictions, target_names=['Normal', 'Phishing']))

# 앙상블 모델 성능 분석
print(f"\n=== 앙상블 모델 상세 성능 ===")
print(classification_report(test_labels, ensemble_predictions, target_names=['Normal', 'Phishing']))

# 클래스별 성능 분석 (앙상블)
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, support = precision_recall_fscore_support(test_labels, ensemble_predictions)
print(f"\n=== 앙상블 클래스별 상세 성능 ===")
print(f"Normal   - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}")
print(f"Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}")

# ROC AUC 계산 (앙상블)
ensemble_probs = np.array(ensemble_probabilities)[:, 1]
fpr, tpr, _ = roc_curve(test_labels, ensemble_probs)
roc_auc = auc(fpr, tpr)
print(f"\n앙상블 테스트 AUC: {roc_auc:.3f}")

# 단일 모델 AUC
single_probs = np.array(test_probabilities)[:, 1]
single_fpr, single_tpr, _ = roc_curve(test_labels, single_probs)
single_roc_auc = auc(single_fpr, single_tpr)
print(f"단일 모델 테스트 AUC: {single_roc_auc:.3f}")

print(f"\n=== 개선 효과 요약 ===")
print(f"모델 복잡도: {sum(p.numel() for p in final_model.parameters()):,} 파라미터")
print(f"K-Fold 검증 안정성: ±{std_acc:.2f}% 표준편차")
print(f"AUC 개선: {single_roc_auc:.3f} → {roc_auc:.3f} (+{roc_auc-single_roc_auc:.3f})")
print(f"Balanced Accuracy: {np.mean([recall[0], recall[1]]):.3f}")

# 결과 시각화

In [ ]:
# 결과 시각화
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Fold별 성능 비교

In [ ]:
# Fold별 성능 비교
folds = [f"Fold {i+1}" for i in range(K_FOLDS)]
ax1.bar(folds, all_val_accs, alpha=0.7, color="skyblue")
ax1.axhline(y=mean_acc, color="red", linestyle="--", label=f"평균: {mean_acc:.2f}%")
ax1.set_title("Fold별 검증 정확도 (클래스 가중치 적용)")
ax1.set_xlabel("Fold")
ax1.set_ylabel("Accuracy (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 테스트 confusion matrix

In [ ]:
# 테스트 confusion matrix
cm = confusion_matrix(test_labels, test_predictions)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax2,
    xticklabels=["Normal", "Phishing"],
    yticklabels=["Normal", "Phishing"],
)
ax2.set_title(f"Test Confusion Matrix (Accuracy: {single_model_accuracy:.2f}%)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")

# ROC Curve

In [ ]:
# ROC Curve
ax3.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
ax3.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.05])
ax3.set_xlabel("False Positive Rate")
ax3.set_ylabel("True Positive Rate")
ax3.set_title("ROC Curve")
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3)

# 클래스별 성능 비교 (Precision, Recall, F1)

In [ ]:
# 클래스별 성능 비교 (Precision, Recall, F1)
metrics = ["Precision", "Recall", "F1-Score"]
normal_scores = [precision[0], recall[0], f1[0]]
phishing_scores = [precision[1], recall[1], f1[1]]

x = np.arange(len(metrics))
width = 0.35

ax4.bar(x - width / 2, normal_scores, width, label="Normal", alpha=0.7, color="blue")
ax4.bar(x + width / 2, phishing_scores, width, label="Phishing", alpha=0.7, color="red")
ax4.set_xlabel("Metrics")
ax4.set_ylabel("Score")
ax4.set_title("클래스별 성능 지표 (불균형 대응)")
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../../datas/analysisData/2nd_model_balanced_results.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# 최종 모델 저장

In [ ]:
# 최종 모델 저장
final_model_path = "../../models/kobert_2nd_model_balanced.pth"
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "model_config": {"hidden_size": 512, "num_classes": 2, "dropout": 0.2},
        "tokenizer_name": MODEL_NAME,
        "class_weights": class_weights.tolist(),
        "kfold_results": {
            "mean_val_acc": mean_acc,
            "std_val_acc": std_acc,
            "fold_accs": all_val_accs,
            "fold_details": fold_results,
        },
        "test_accuracy": single_model_accuracy,
        "test_auc": single_roc_auc,
        "ensemble_accuracy": ensemble_accuracy,
        "ensemble_auc": roc_auc,
        "balanced_accuracy": np.mean([recall[0], recall[1]]),
    },
    final_model_path,
)

print(f"\n최종 모델이 저장되었습니다: {final_model_path}")

# 결과 요약

In [ ]:
# 결과 요약
print("\n" + "=" * 60)
print("2차 모델 (클래스 불균형 대응) 완료 요약")
print("=" * 60)
print(f"모델 아키텍처: KoBERT + LSTM + Attention (화자 정보 제거)")
print(f"교차검증: {K_FOLDS}-Fold Stratified")
print(f"훈련 데이터: {len(train_dialogues)} 대화 (merged_labeled_data.csv)")
print(f"테스트 데이터: {len(test_dialogues)} 샘플 (1차모델_테스트데이터셋.csv)")
print(f"")
print(f"=== 클래스 불균형 대응 ===")
print(f"클래스 가중치: Normal={class_weights[0]:.3f}, Phishing={class_weights[1]:.3f}")
print(f"")
print(f"=== 성능 결과 ===")
print(f"K-Fold 평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"단일 모델 테스트 정확도: {single_model_accuracy:.2f}%")
print(f"앙상블 모델 테스트 정확도: {ensemble_accuracy:.2f}%")
print(f"단일 모델 테스트 AUC: {single_roc_auc:.3f}")
print(f"앙상블 모델 테스트 AUC: {roc_auc:.3f}")
print(f"Balanced Accuracy: {np.mean([recall[0], recall[1]]):.3f}")
print(f"")
print(f"=== 클래스별 성능 ===")
print(
    f"Normal   - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}"
)
print(
    f"Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}"
)
print(f"")
print(
    f"성능 안정성: 표준편차 {std_acc:.2f}%로 {'안정적' if std_acc < 2.0 else '다소 불안정'}"
)
print("앙상블을 통한 성능 향상 및 안정성 확보!")
print("=" * 60)